# Обработка больших данных с Apache Spark Часть 1

## Практика

In [57]:
# установить Spark
!pip install pyspark


In [ ]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Создаем Spark сессию
spark = SparkSession.builder \
    .appName("Calls Analysis") \
    .config("spark.sql.repl.eagerEval.enabled", True) \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

In [50]:
spark

## При установке может потребоваться установить JAVA
https://www.oracle.com/java/technologies/downloads/#jdk25-windows


In [ ]:
df = spark.read.csv('calls_sample_data_homework.csv', header=True)

In [52]:
df.printSchema()

root
 |-- call_id: string (nullable = true)
 |-- subscriber_id: string (nullable = true)
 |-- call_start: string (nullable = true)
 |-- call_end: string (nullable = true)
 |-- region_code: string (nullable = true)
 |-- tariff_name: string (nullable = true)
 |-- monthly_fee: string (nullable = true)



In [6]:
# Исправление типов данных
df = (df.withColumn("monthly_fee",col("monthly_fee").cast("float"))
      .withColumn("call_start",col("call_start").cast("timestamp"))
      .withColumn("call_end",col("call_end").cast("timestamp"))
)

In [7]:
df.printSchema()

root
 |-- call_id: string (nullable = true)
 |-- subscriber_id: string (nullable = true)
 |-- call_start: timestamp (nullable = true)
 |-- call_end: timestamp (nullable = true)
 |-- region_code: string (nullable = true)
 |-- tariff_name: string (nullable = true)
 |-- monthly_fee: float (nullable = true)



## Изучите примеры задач и добавьте недостающие решения
#### К некоторым задачам уже даны решения, используйте их как шаблон

#### Все задачи необходимо решить в двух вариантах PySpark и SparkSQL

## Если ресурсы системы не позволяют запустить Spark, запустите файл в Google Colab

## В качестве ответа приложите данный файл в форму сдачи

## (!) Не удаляйте выходные данные ячеек


### Задача 1: Подсчитать общее количество звонков

In [53]:
# Создание представления
df.createOrReplaceTempView('calls')

**PySpark:**

In [ ]:
# Пример
df.count()

1000

**SparkSQL:**

In [ ]:
# Пример
spark.sql("""
SELECT COUNT(*) as total_calls 
FROM calls
""")


total_calls
1000


### Задача 2: Найти количество уникальных абонентов

**PySpark:**

In [ ]:
# Пример
df.select("subscriber_id").distinct().count()

605

**SparkSQL:**

### Задача 3: Найти среднюю продолжительность звонка в минутах

**PySpark:**

In [ ]:
# Пример
df=df.withColumn(
    "duration", 
    expr("(unix_timestamp(call_end) - unix_timestamp(call_start)) ")
)

In [14]:
avg_duration = df.select(avg("duration")).collect()[0][0]
print(f"Средняя продолжительность звонка = {avg_duration/60:.2f} минут")

Средняя продолжительность звонка = 17.45 минут


**SparkSQL:**

In [ ]:
# Пример
spark.sql("""
SELECT 
    AVG((UNIX_TIMESTAMP(call_end) - UNIX_TIMESTAMP(call_start)) / 60) as avg_duration_minutes
FROM calls
""")

avg_duration_minutes
17.453966666666663


### Задача 4: Найти топ-5 самых активных абонентов по количеству звонков

**PySpark:**

In [ ]:
# Пример
(df.groupBy("subscriber_id") 
    .agg(count("*").alias("call_count")) 
    .orderBy(desc("call_count")) 
    .limit(5))

subscriber_id,call_count
00000293,6
00000790,6
00000757,6
00000495,5
00000364,5


**SparkSQL:**

### Задача 5: Найти распределение звонков по тарифам

**PySpark:**

**SparkSQL:**

### Задача 6: Найти самый популярный час для звонков

**PySpark:**

In [ ]:
# hour("call_start")


**SparkSQL:**

### Задача 9: Найти количество звонков по регионам

**PySpark:**

**SparkSQL:**

### Задача 10: Найти среднюю абонентскую плату

**PySpark:**

**SparkSQL:**

### Задача 11: Найти абонентов с самым дорогим тарифом

**PySpark:**

**SparkSQL:**

### Задача 12: Найти количество звонков по дням недели

**PySpark:**

In [ ]:
# Пример

day_calls_1 = (df.withColumn("day_of_week", date_format("call_start", "EEEE"))
                .withColumn("day_of_week_number", dayofweek("call_start"))
                .groupBy("day_of_week", "day_of_week_number").agg(count("*").alias("call_count"))
                .sort('day_of_week_number'))

print("Количество звонков по дням недели")
day_calls_1.show()


Количество звонков по дням недели
+-----------+------------------+----------+
|day_of_week|day_of_week_number|call_count|
+-----------+------------------+----------+
|     Sunday|                 1|       130|
|     Monday|                 2|       134|
|    Tuesday|                 3|       138|
|  Wednesday|                 4|       122|
|   Thursday|                 5|       205|
|     Friday|                 6|       132|
|   Saturday|                 7|       139|
+-----------+------------------+----------+



**SparkSQL:**

### Задача 13: Найти среднюю продолжительность звонка по тарифам

**PySpark:**

**SparkSQL:**

### Задача 14: Найти абонентов, которые совершили более 5 звонков

**PySpark:**

In [ ]:
# Пример
active_subscribers_1 = df.groupBy("subscriber_id") \
    .agg(count("*").alias("call_count")) \
    .filter(col("call_count") > 5) \
    .orderBy(desc("call_count"))

print("Абоненты с более чем 5 звонками")
active_subscribers_1.show(10)


Абоненты с более чем 5 звонками
+-------------+----------+
|subscriber_id|call_count|
+-------------+----------+
|     00000790|         6|
|     00000293|         6|
|     00000757|         6|
+-------------+----------+



**SparkSQL:**

### Задача 15: Найти распределение звонков по времени суток

**PySpark:**

In [ ]:
# Пример
time_dist_1 = df.withColumn("hour", hour("call_start")) \
    .withColumn("time_of_day",
        when((col("hour") >= 0) & (col("hour") < 6), "Ночь (00:00-06:00)")
        .when((col("hour") >= 6) & (col("hour") < 12), "Утро (06:00-12:00)")
        .when((col("hour") >= 12) & (col("hour") < 18), "День (12:00-18:00)")
        .otherwise("Вечер (18:00-00:00)")
    ).groupBy("time_of_day").agg(count("*").alias("call_count"))

time_dist_1.show()



**SparkSQL:**

### Задача 16: Найти максимальную продолжительность звонка

**PySpark:**

**SparkSQL:**

### Задача 17: Найти звонки, которые длились менее 1 минуты

**PySpark:**

**SparkSQL:**

### Задача 18: Найти среднее количество звонков на абонента

**PySpark:**

**SparkSQL:**

### Задача 19: Найти регион с наибольшей средней продолжительностью звонка

**PySpark:**

**SparkSQL:**

### Задача 20: Найти день с наибольшим количеством звонков

**PySpark:**

**SparkSQL:**

In [47]:

# Остановка Spark сессии
spark.stop()